# Same exact matchup logic as 01_match_maker.ipynb, but just meant for one day.
# Goal is the strip plots seen at the bottom.

In [ ]:
import os
import sys
import glob
import shutil
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd


import geopandas as gpd
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import LineString, Polygon
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis, pearsonr


import xarray as xr
import earthaccess as ea
from sliderule import icesat2, sliderule
from harmony import (
    BBox,
    CapabilitiesRequest,
    Client,
    Collection,
    JobsRequest,
    LinkType,
    Request,
)


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import contextily as cx
import seaborn as sns
from matplotlib.colors import LogNorm, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


from aux_fx_process import (
    grid_data,
    add_lon_lat,
    get_photons,
    water_mask,
    water_surface_retrieval_and_binning,
    get_segment_vars,
    get_PACE_time_windows,
    get_PACE_data,
    gridded_chl,
    get_segment_centers,
    match_chl_to_segments,
)
from aux_fx_plot import ATLAS_overlays_TX, make_scatter_plot


ea.login()
# ea.login(strategy="interactive", persist=True)

In [ ]:
"""

Configure Study Area + Photon-Input Parameters

"""
# Initialize sliderule
sliderule.init("slideruleearth.io", verbose=True)
pd.set_option("display.max_columns", 80)
[[[-97.5461279749,25.9114436991],
  [-93.0678475338,25.9114436991],
  [-93.0678475338,30.1923463],
  [-97.5461279749,30.1923463],
  [-97.5461279749,25.9114436991]]]

LAT_MIN = 25.911
LAT_MAX = 30.192
LON_MIN = -97.546
LON_MAX = -93.067
# from imports import params
# aoi, resources, base_parms = params()
aoi = [
        {"lat": LAT_MIN, "lon": LON_MIN},
        {"lat": LAT_MIN, "lon": LON_MAX},
        {"lat": LAT_MAX, "lon": LON_MAX},
        {"lat": LAT_MAX, "lon": LON_MIN},
        {"lat": LAT_MIN, "lon": LON_MIN},
    ]

resources = None


# FOR STRIP: change the date "t0" for desired day

In [ ]:
t0 = "2025-08-12"
t0_object = datetime.strptime(t0, "%Y-%m-%d")

# FOR STRIP: Loads in matchups for one day. and Saves. (< 1 minute runtime)


In [ ]:

""" IMPORT FX """
# from aux_fx_process import add_lon_lat
from aux_fx_process import get_photons
from aux_fx_process import water_mask
from aux_fx_process import water_surface_retrieval_and_binning
from aux_fx_process import get_segment_vars
from aux_fx_process import get_PACE_time_windows
from aux_fx_process import get_PACE_data
from aux_fx_process import gridded_chl
# from aux_fx_process import get_segment_centers
from aux_fx_process import match_chl_to_segments
import sys

current = t0_object
TIMEDELT = 1
LAND = gpd.read_file("ne_10m_land.shp")  
SEGMENT_LENGTH = 1000
TOLERANCE_DEG = 0.02
TOLERANCE_TIME = "12h"

""" Set up window dates """
# 
window_start = current
window_start_str = window_start.strftime("%Y-%m-%d")
window_end = current + timedelta(days=TIMEDELT)
window_end_str = window_end.strftime("%Y-%m-%d")
print(window_start.strftime("%Y-%m-%d"),
    window_end.strftime("%Y-%m-%d"))



""" Get photons """

photons_xy = get_photons(window_start_str, window_end_str, aoi)
if photons_xy is None:
    print(f"No photons | OR SLIDERULE FAIL for {window_start_str} - {window_end_str}, skipping window.")
    current += timedelta(days=TIMEDELT)
    sys.exit()



""" Mask Data over water """

photons_xy_ocean = water_mask(photons_xy, LAND)



""" Water Surface Retrieval """

water_surface, below_surface = water_surface_retrieval_and_binning(photons_xy_ocean, SEGMENT_LENGTH)
if len(below_surface) == 0:
    print(f"No subsurface photons for {window_start_str} - {window_end_str}, skipping window.")
    current += timedelta(days=TIMEDELT)
    sys.exit()



""" Segment Variable Retrieval """

segment_vars = get_segment_vars(below_surface, water_surface, THRESHOLD_LIMIT)
if len(segment_vars) == 0:
        print(f"No segments passed THRESHOLD_LIMIT for {window_start_str} - {window_end_str}, skipping window.")
        current += timedelta(days=4)
        sys.exit()



""" Get Time Windows for PACE Data """

merged_windows_df = get_PACE_time_windows(segment_vars)



""" Get PACE Data (uncomment out if already done once)
            returns directory for this PACE window data too """
harmony_client = Client(token=ea.get_edl_token())
PACE_ROOT = "DATA/PACE_l2_Texas/"
os.makedirs(PACE_ROOT, exist_ok=True)
pace_window_dir = get_PACE_data(merged_windows_df, window_start_str, window_end_str, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, harmony_client, PACE_ROOT)



""" Form a gridded geodataframe of PACE chl """

chl_a_gdf = gridded_chl(pace_window_dir)
if chl_a_gdf is None:
    print(f"No usable PACE granules for {window_start_str} - {window_end_str}, skipping window.")
    shutil.rmtree(pace_window_dir, ignore_errors=True)
    current += timedelta(days=TIMEDELT)
    sys.exit()



""" Match PACE Chl to ATLAS segments: """ 

tolerance_deg = TOLERANCE_DEG
tolerance_time = TOLERANCE_TIME
matchup_df = match_chl_to_segments(segment_vars, chl_a_gdf, tolerance_deg, tolerance_time)
matchup_df['chl_a'].describe()



"""  Filter the matchup dataframe to only keep rows where PACE chl_a is NOT NaN """ 

matchup_df_filtered = matchup_df[matchup_df['chl_a'].notna()]



""" Print the descriptive statistics """ 

print(matchup_df_filtered['chl_a'].describe())
matchup_df
print(matchup_df_filtered['lidar_date'].value_counts().head())
matchup_df_filtered



""" SAVE MATCHUPS """ 

os.makedirs("MATCHUPS/GULF_DATA_INPACEGAPS", exist_ok=True)  
os.makedirs("MATCHUPS/GULF_DATA_FILTERED", exist_ok=True) 
matchup_df.to_parquet("MATCHUPS/GULF_DATA_INPACEGAPS/data._" + window_start_str + "_" + window_end_str + "matchups_GULF_INPASSIVEGAPS")
matchup_df_filtered.to_parquet("MATCHUPS/GULF_DATA_FILTERED/data._" + window_start_str + "_" + window_end_str + "matchups_GULF")



""" deletes PACE data now that Matchups downloaded (toggle off to save space) """ 

shutil.rmtree(pace_window_dir, ignore_errors=True)
print(f"Cleaned up {pace_window_dir}")    



# Begin Plots


In [ ]:
""" LOAD IN DATA MATCHUS """

matchups = gpd.read_parquet("MATCHUPS/GULF_DATA_INPACEGAPS/data._2025-08-12_2025-08-13matchups_GULF_INPASSIVEGAPS")
matchups_filtered = gpd.read_parquet("MATCHUPS/GULF_DATA_FILTERED/data._2025-08-12_2025-08-13matchups_GULF")
matchups_filtered.columns

In [ ]:
""" PREDICT CHL """
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import joblib

features_trimmed = ['D10', 'D50', 'N_surface', 'N_subsurface', 'log_R_sw', 'z_max', 'std_of_depth', 'D75/D25']

# apply the already-trained RF model onto the LiDAR data for the day
def apply_model(df, model, features_trimmed):
    df = df.copy()
    df["log_R_sw"] = np.log10(df["R_sw"])
    df["D75/D25"] = df["D75"] / df["D25"]
    df["pred_log_chl_a"] = model.predict(df[features_trimmed])
    df["pred_chl_a"] = 10 ** df["pred_log_chl_a"]
    return df

rf_trimmed = joblib.load('RF_JULY14_TRIMMEDFEATS.joblib')

matchups = apply_model(matchups, rf_trimmed, features_trimmed)
matchups_filtered = apply_model(matchups_filtered, rf_trimmed, features_trimmed)
# unfiltered_matchups = apply_model(segment_vars, rf_trimmed, features_trimmed)

In [ ]:
""" Plot strip plots """

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.cm as cm
import matplotlib.colors as colors



def strip_plot_raw_track_vs_pace(df_raw_track, df_filtered_matchups, vmin=0.01, vmax=10.0,
                                 bin_size_m=1000, title=None, save_path=None):
    
    COLORMAP = "viridis"

    # Clean and bin the RAW ATLAS track (1000m)
    lidar_track = df_raw_track.dropna(subset=["x_atc"]).copy()

    # bin in blocks of 1 km.
    lidar_track["x_atc_bin"] = (lidar_track["x_atc"] // bin_size_m) * bin_size_m

    """ filtered LiDAR data binned """
    LIDAR_filt_and_bin = lidar_track.groupby("x_atc_bin")["pred_chl_a"].mean()

    # Clean and bin the PACE observations. Same logic as for ATLAS, just maeke sure to drop any chl_a NaN's from PACE that slipped through
    d_filt = df_filtered_matchups.dropna(subset=["x_atc", "chl_a"]).copy()
    d_filt["x_atc_bin"] = (d_filt["x_atc"] // bin_size_m) * bin_size_m

    """ filtered PACE data binned """
    PACE_filt_and_bin = d_filt.groupby("x_atc_bin")["chl_a"].mean()

    # Create full bins
    min_bin = lidar_track["x_atc_bin"].min()
    max_bin = lidar_track["x_atc_bin"].max()
    full_bins = np.arange(min_bin, max_bin + bin_size_m, bin_size_m)
    
    # Align and reindex to bins
    df_plot = pd.DataFrame(index=full_bins)
    df_plot["pred_chl_a"] = LIDAR_filt_and_bin.reindex(full_bins)  # ATLAS (Continuous)
    df_plot["chl_a"] = PACE_filt_and_bin.reindex(full_bins)        # PACE (Gapped)
    
    # change index to x_atc_bin
    df_plot = df_plot.reset_index().rename(columns={"index": "x_atc_bin"})
    
    # Subtract min_bin so the x-axis starts at 0
    df_plot["dist_km"] = (df_plot["x_atc_bin"] - min_bin) / 1000.0

    # Set spatial extent (this will now run from 0 to max_distance)
    norm = LogNorm(vmin=vmin, vmax=vmax)
    extent = [df_plot["dist_km"].min(), df_plot["dist_km"].max(), 0, 1]




    """ Plotting """
    fig, axes = plt.subplots(3, 1, figsize=(12, 3), sharex=True, constrained_layout=True)

    # Top: ATLAS track. Model-derived chl-a
    axes[0].imshow(df_plot["pred_chl_a"].values[np.newaxis, :], aspect="auto", cmap=COLORMAP, norm=norm, extent=extent)
    axes[0].set_yticks([])
    axes[0].set_ylabel("ATLAS",fontsize=10)

    # Middle: PACE observations
    axes[1].imshow(df_plot["chl_a"].values[np.newaxis, :], aspect="auto", cmap=COLORMAP, norm=norm, extent=extent)
    axes[1].set_yticks([])
    axes[1].set_ylabel("PACE", fontsize=10)


    if title:
        fig.suptitle(title, fontsize=12)

    
    # Percent difference (ATLAS relative to PACE)
    df_plot["pct_diff"] = 100 * (
        (df_plot["chl_a"] - df_plot["pred_chl_a"]) /
        df_plot["chl_a"]
    )
    
    # avoid any infinities in case chl_a == 0
    df_plot.replace([np.inf, -np.inf], np.nan, inplace=True)
    pct_norm = colors.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)

    # Bottom: % Diff
    im = axes[2].imshow(
        df_plot["pct_diff"].values[np.newaxis, :],
        aspect="auto",
        cmap="RdBu_r",
        norm=pct_norm,
        extent=extent,
    )
    axes[2].set_yticks([])
    axes[2].set_ylabel("Difference\n[%]", fontsize=10)
    axes[2].set_xlabel("Along-track distance [km]")


    # Chl-a colorbar
    cax1 = fig.add_axes([1.02, 0.15, 0.02, 0.84])   # left, bottom, width, height
    cbar1 = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=COLORMAP),
        cax=cax1
    )
    cbar1.set_label("Chl-a [mg m$^{-3}$]")


    # Percent difference colorbar
    cax2 = fig.add_axes([1.12, 0.15, 0.02, 0.84])
    cbar2 = fig.colorbar(im, cax=cax2)
    # cbar2.set_label("Difference [%]")
    cbar2.set_label(r"$[\mathrm{PACE}-\mathrm{ATLAS}]$ / PACE")

    # Plot parameters
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 8,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
    })

    """ SET LIMITS """ # Will be necessary
    
    axes[0].set_xlim([0, 205])
    axes[1].set_xlim([0, 205])
    axes[2].set_xlim([0, 205])
    
    # axes[0].set_xlim([14035, None])
    # axes[1].set_xlim([14035, None])
    # axes[2].set_xlim([14035, None])
    

    if save_path:
        fig.savefig(save_path, dpi=600, bbox_inches="tight", facecolor="white")
    return fig, axes, df_plot

fig, axes, d = strip_plot_raw_track_vs_pace(
    df_raw_track=matchups,      
    df_filtered_matchups=matchups_filtered,      
    bin_size_m=1000, 
    title="", 
    save_path="FINAL_TX_FIGS/percentdiff_strip_matchups_2025-08-12_left.png"
)

